In [11]:
import duckdb
import pandas as pd
from datetime import datetime

In [12]:
conn = duckdb.connect('../databases/data.db', read_only=False)

In [14]:
df = conn.execute(f"""
                  SELECT 
                  * FROM (
                  SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY date_ingestion DESC) AS row
                  FROM bronze_z0019
                  WHERE date_ingestion >= '{datetime.now().date()}' )
                  WHERE row = 1
""").fetchdf()
df.head(df.size)

,NATBR,MAKTX,WERKS,MAINS,LABST,file_name,date_ingestion,row
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-07-08 11:37:07,1
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-07-08 11:37:07,1
2,10004,CHAVE DE FENDA,BT50,100,200,z0019_2.csv,2026-07-08 11:37:40,1
3,10005,ALICATE,BT10,100,300,z0019_2.csv,2026-07-08 11:37:40,1
4,10003,PREGO,BT10,100,60,z0019_2.csv,2026-07-08 11:37:40,1


In [15]:
df_final = df.drop(columns=['row', 'date_ingestion', 'file_name'])

df_final = df_final.rename(columns={'NATBR': 'id', 'MAKTX': 'nm_product', 'WERKS': 'id_category', 'MAINS': 'id_supplier', 'LABST': 'price'})
df_final.head(df.size)

,id,nm_product,id_category,id_supplier,price
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10004,CHAVE DE FENDA,BT50,100,200
3,10005,ALICATE,BT10,100,300
4,10003,PREGO,BT10,100,60


In [17]:
df2 = df_final

df2.head(df2.size)

,id,nm_product,id_category,id_supplier,price
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10004,CHAVE DE FENDA,BT50,100,200
3,10005,ALICATE,BT10,100,300
4,10003,PREGO,BT10,100,60


In [18]:
df2 = df2.astype({
    'id': int,
    'nm_product': str,
    'id_category': str,
    'id_supplier': int,
    'price': float
})

In [19]:
df2.dtypes

id               int64
nm_product         str
id_category        str
id_supplier      int64
price          float64
dtype: object

In [20]:
df2.head(10)

,id,nm_product,id_category,id_supplier,price
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,CHAVE DE FENDA,BT50,100,200.0
3,10005,ALICATE,BT10,100,300.0
4,10003,PREGO,BT10,100,60.0


In [21]:
conn.execute("""
CREATE TABLE IF NOT EXISTS silver_products (
             id BIGINT,
             nm_product TEXT,
             id_category TEXT,
             id_supplier BIGINT,
             price FLOAT)
""")

In [23]:
conn.execute("""insert into silver_products select * from df2""")

In [24]:
df_result = conn.execute("""select * from silver_products""").fetchdf()
df_result.head(10)

,id,nm_product,id_category,id_supplier,price
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,CHAVE DE FENDA,BT50,100,200.0
3,10005,ALICATE,BT10,100,300.0
4,10003,PREGO,BT10,100,60.0


In [25]:
conn.close()